# AI-Based Robotic Fruit Sorting System
## LOCAL TRAINING VERSION — Runs directly on your Mac
### Dataset: Mendeley Original Image dataset (already in Downloads folder)

### SECTION 1 — Environment Setup

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

os.makedirs('./exported_models', exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('Setup complete!')

### SECTION 2 — Dataset Filtering (Copy Only Required Classes Locally)

In [ ]:
import shutil

# Dataset is already extracted in your Downloads folder
SOURCE_DIR = '/Users/saakshi/Downloads/Original Image'
DATASET_DIR = './dataset'

# The 6 required classes ONLY (from rulebook)
REQUIRED_CLASSES = [
    'FreshApple', 'RottenApple',
    'FreshGrape', 'RottenGrape',
    'FreshStrawberry', 'RottenStrawberry'
]

# Copy only required folders to local ./dataset (skip if already done)
if not os.path.exists(DATASET_DIR):
    print('Copying required classes to local ./dataset folder...')
    os.makedirs(DATASET_DIR, exist_ok=True)
    for cls in REQUIRED_CLASSES:
        src = os.path.join(SOURCE_DIR, cls)
        dst = os.path.join(DATASET_DIR, cls)
        if os.path.exists(src):
            shutil.copytree(src, dst)
            print(f'  Copied: {cls}')
        else:
            print(f'  WARNING: {cls} not found in source!')
    print('Done copying!')
else:
    print('Dataset folder already exists. Skipping copy.')

# Verify counts
print('\nImage counts per class:')
for cls in REQUIRED_CLASSES:
    cls_path = os.path.join(DATASET_DIR, cls)
    if os.path.exists(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f'  {cls}: {count} images')
    else:
        print(f'  {cls}: MISSING!')

### SECTION 3 — Dataset Validation (Remove Corrupt Images)

In [ ]:
from PIL import Image
from pathlib import Path

print('Validating images...')
total_removed = 0
for cls in REQUIRED_CLASSES:
    cls_path = Path(DATASET_DIR) / cls
    if not cls_path.exists(): continue
    images = list(cls_path.glob('*.jpg')) + list(cls_path.glob('*.jpeg')) + list(cls_path.glob('*.png'))
    for img_path in images:
        try:
            img = Image.open(img_path)
            img.verify()
        except Exception:
            os.remove(img_path)
            total_removed += 1
            print(f'  Removed corrupt: {img_path.name}')
print(f'Validation complete. Removed {total_removed} corrupt images.')

### SECTION 4 — Train / Validation / Test Split

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = val_ds
ALLOWED_CLASSES = train_ds.class_names
print('Classes Detected:', ALLOWED_CLASSES)

### SECTION 5 — Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2)
])
print('Augmentation pipeline ready.')

### SECTION 6 — TensorFlow Data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare_pipeline(ds, augment=False):
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds = prepare_pipeline(train_ds, augment=True)
val_ds = prepare_pipeline(val_ds)
test_ds = prepare_pipeline(test_ds)
print('Data pipeline ready.')

### SECTION 7 — CNN Architecture

In [ ]:
model = Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(6, activation='softmax')
])

model.summary()

### SECTION 8 — Compile and Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint('./exported_models/best_model.keras', save_best_only=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping, checkpoint]
)

### SECTION 9 — Visualize Training

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

### SECTION 10 — Evaluate on Test Set

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f'Test Accuracy: {accuracy*100:.2f}%')

### SECTION 11 — Unknown Fruit Detection Logic

In [ ]:
def predict_fruit(img_array, model, class_names, threshold=0.7):
    predictions = model.predict(img_array)
    max_confidence = np.max(predictions)
    if max_confidence < threshold:
        return 'Unknown Fruit', max_confidence
    else:
        predicted_class = class_names[np.argmax(predictions)]
        return predicted_class, max_confidence

print('Inference function ready.')

### SECTION 12 — Export Models (.keras, SavedModel, .tflite, class_names.json)

In [ ]:
import json

# Save .keras format (Keras 3 compatible)
model.save('./exported_models/final_model.keras')
print('Saved: final_model.keras')

# Save SavedModel format (for TFLite/TFServing)
model.export('./exported_models/final_model_saved')
print('Saved: final_model_saved/')

# TFLite Export
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('./exported_models/model.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved: model.tflite')

# Class names
with open('./exported_models/class_names.json', 'w') as f:
    json.dump(ALLOWED_CLASSES, f)
print('Saved: class_names.json')
print('\nAll exports complete!')

### SECTION 13 — ESP32 Integration Hooks

In [ ]:
import requests

ESP32_IP = '192.168.4.1'

def trigger_sorting_arm(predicted_class):
    if 'Fresh' in predicted_class:
        angles = {'base': 120, 'shoulder': 90, 'elbow': 45, 'claw': 10}
    elif 'Rotten' in predicted_class:
        angles = {'base': 60, 'shoulder': 90, 'elbow': 45, 'claw': 10}
    else:
        print('Unknown fruit - no action.')
        return
    url = f"http://{ESP32_IP}/move?base={angles['base']}&shoulder={angles['shoulder']}&elbow={angles['elbow']}&claw={angles['claw']}"
    try:
        print(f'Sending to ESP32: {url}')
        # response = requests.get(url, timeout=2)
    except Exception as e:
        print('ESP32 Error:', e)

print('ESP32 integration hooks ready.')